# DeLeaker quickstart

Generate a single image with attention-rewriting to suppress entity leakage.

**Prerequisites**
1. `pip install -e .` from the repo root.
2. FLUX.1-dev is gated — accept the license at https://huggingface.co/black-forest-labs/FLUX.1-dev and run `huggingface-cli login` once.
3. CUDA GPU with ~16 GB VRAM at 512x512 (~24 GB at 1024x1024).

In [ ]:
import torch
from deleaker import DeleakerFluxPipeline

PROMPT = "A bat and an owl are perched side by side on a tree branch."
ENTITIES = ["bat", "owl"]
SEED = 200

## Load the pipeline

First call downloads the model (~24 GB) into the local Hugging Face cache.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
pipe = DeleakerFluxPipeline.from_pretrained(
    "black-forest-labs/FLUX.1-dev", torch_dtype=torch.float16
).to(device)

## Generate

Each entity in `entities` must appear verbatim in `prompt`. Pass `use_deleaker=False` to compare against the vanilla baseline at the same seed.

In [ ]:
image = pipe(
    prompt=PROMPT,
    entities=ENTITIES,
    num_inference_steps=20,
    guidance_scale=3.5,
    height=512,
    width=512,
    generator=torch.Generator(device=device).manual_seed(SEED),
    max_sequence_length=256,
).images[0]
image

In [ ]:
image.save("out.png")